# Stage-Based Modeling Walkthrough

This notebook explains and runs the Gate 0, Gate 1, Gate 2, and Axis B modeling scripts. The scripts remain the canonical source for reproduced outputs; this notebook is a readable companion.

## 1. Load data and inspect gate samples

In [1]:
from pathlib import Path
import sys

# Resolve repository root even when Jupyter starts in a different working directory.
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "RQ2_Prompt_Effectiveness_Modeling").exists() and (candidate / "Dataset_Construction").exists():
        ROOT = candidate
        break
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
from RQ2_Prompt_Effectiveness_Modeling.analysis.common import load_analysis_dataset

df = load_analysis_dataset(ROOT)
df.head()

,Case ID,PR_Link,Conversation_Link,Outcome_Class,Context,Specificity,Verification,Rationale,PQS,PR_Size,...,Case_ID,Repository,PR_Number,Merged,Closed,Generated_Code,Adopted_Code,Resolved,Close_Event,Merge_Event
0,PA-1,https://github.com/Altinn/altinn-broker/pull/259,https://chat.openai.com/share/b7853f70-84b8-47...,PA,1,1,0,Context was scored 1 because the prompt provid...,2,125.0,...,PA-1,Altinn/altinn-broker,259,1,0,1,1,1,0,1
1,PA-2,https://github.com/Hochfrequenz/kohlrahbi/pull...,https://chat.openai.com/share/4ad4c1ad-6f13-4a...,PA,1,1,0,Context was scored 1 because the prompt provid...,2,51.0,...,PA-2,Hochfrequenz/kohlrahbi,158,1,0,1,1,1,0,1
2,PA-3,https://github.com/MartinsOnuoha/what-should-i...,https://chat.openai.com/share/2aa6268a-7a4e-47...,PA,2,2,2,Context was scored 2 because the prompt mentio...,6,300.0,...,PA-3,MartinsOnuoha/what-should-i-design,8,1,0,1,1,1,0,1
3,PA-4,https://github.com/Opetushallitus/ludos/pull/102,https://chat.openai.com/share/bdfcb857-08a3-4f...,PA,1,1,1,Context was scored 1 because the prompt provid...,3,620.0,...,PA-4,Opetushallitus/ludos,102,1,0,1,1,1,0,1
4,PA-5,https://github.com/SharezoneApp/sharezone-app/...,https://chat.openai.com/share/fd82b66d-d949-43...,PA,2,1,1,Context was scored 2 because the prompt mentio...,4,18.0,...,PA-5,SharezoneApp/sharezone-app,980,1,0,1,1,1,0,1


In [2]:
gate0 = df[df.Outcome_Class.isin(['PA','PN','NE'])]
gate1 = df[df.Outcome_Class.isin(['PA','PN'])]
gate2 = df[df.Outcome_Class.eq('PA')]
axisb = df.dropna(subset=['Time_To_Event'])
pd.DataFrame({'stage':['Gate 0','Gate 1','Gate 2','Axis B'], 'n':[len(gate0), len(gate1), len(gate2), len(axisb)]})

,stage,n
0,Gate 0,222
1,Gate 1,142
2,Gate 2,89
3,Axis B,261


## 2. Gate 0: code generation

Gate 0 models whether a prompt produces actionable code, contrasting NE against PA/PN cases.

In [3]:
from RQ2_Prompt_Effectiveness_Modeling.analysis.quantitative import gate0_generation
gate0_results = gate0_generation.run(ROOT)
gate0_results

,Model,Variable,OR,CI_Low,CI_High,p_value,Formatted
0,(1) Baseline,Context (C),2.162901,1.141098,4.099681,0.018053,"2.16* [1.14, 4.10]"
1,(1) Baseline,Specificity (S),62.962867,8.522027,465.185429,0.000049,"62.96*** [8.52, 465.19]"
2,(1) Baseline,Verification (V),0.902796,0.434277,1.876774,0.784182,"0.90 [0.43, 1.88]"
3,(1) Baseline,Observations,218.000000,NaN,NaN,NaN,218
4,(2) + PR Size,Context (C),2.143088,1.128368,4.070328,0.019861,"2.14* [1.13, 4.07]"
5,(2) + PR Size,Specificity (S),65.838382,8.898159,487.144847,0.000041,"65.84*** [8.90, 487.14]"
6,(2) + PR Size,Verification (V),0.903855,0.433619,1.884035,0.787359,"0.90 [0.43, 1.88]"
7,(2) + PR Size,Log(PR Size),1.115435,0.913939,1.361354,0.282517,"1.12 [0.91, 1.36]"
8,(2) + PR Size,Observations,218.000000,NaN,NaN,NaN,218


## 3. Gate 1: code adoption

Gate 1 is restricted to PA/PN cases and models whether generated code was adopted.

In [4]:
from RQ2_Prompt_Effectiveness_Modeling.analysis.quantitative import gate1_adoption
gate1_results = gate1_adoption.run(ROOT)
gate1_results

,Model,Variable,OR,CI_Low,CI_High,p_value,Formatted
0,(1) Baseline,Context (C),2.284000,1.023582,5.096473,0.043708,"2.28* [1.02, 5.10]"
1,(1) Baseline,Specificity (S),0.606206,0.232033,1.583767,0.306997,"0.61 [0.23, 1.58]"
2,(1) Baseline,Verification (V),7.781273,3.397202,17.822966,0.000001,"7.78*** [3.40, 17.82]"
3,(1) Baseline,Observations,141.000000,NaN,NaN,NaN,141
4,(2) + PR Size,Context (C),2.218866,0.966398,5.094557,0.060195,"2.22 [0.97, 5.09]"
5,(2) + PR Size,Specificity (S),0.792625,0.292440,2.148319,0.647790,"0.79 [0.29, 2.15]"
6,(2) + PR Size,Verification (V),8.454325,3.497812,20.434375,0.000002,"8.45*** [3.50, 20.43]"
7,(2) + PR Size,Log(PR Size),1.361307,1.081919,1.712841,0.008494,"1.36** [1.08, 1.71]"
8,(2) + PR Size,Observations,141.000000,NaN,NaN,NaN,141


## 4. Gate 2: integration depth

Gate 2 is restricted to PA cases and models the fraction of generated code retained in the final implementation.

In [5]:
from RQ2_Prompt_Effectiveness_Modeling.analysis.quantitative import gate2_integration
gate2_results = gate2_integration.run(ROOT)
gate2_results

,Variable,AME,CI_Low,CI_High,p_value,Formatted
0,Context (C),0.124720,0.016083,0.233358,0.024441,"0.125* [0.016, 0.233]"
1,Specificity (S),-0.013427,-0.131396,0.104542,0.823474,"-0.013 [-0.131, 0.105]"
2,Verification (V),-0.008189,-0.105781,0.089403,0.869371,"-0.008 [-0.106, 0.089]"
3,Log(PR Size),-0.035967,-0.074327,0.002393,0.066104,"-0.036 [-0.074, 0.002]"
4,Observations,89.000000,NaN,NaN,NaN,89


## 5. Axis B: lifecycle outcomes

Axis B keeps pull-request lifecycle outcomes separate from prompt-level code generation/adoption/integration effects.

In [6]:
from RQ2_Prompt_Effectiveness_Modeling.analysis.quantitative import axisB_lifecycle
axisb_results = axisB_lifecycle.run(ROOT)
axisb_results

,Model,Variable,HR,CI_Low,CI_High,p_value,Formatted
0,Merge Hazard,Context (C),1.021064,0.799764,1.303598,0.867175,"1.02 [0.80, 1.30]"
1,Merge Hazard,Specificity (S),1.070307,0.814531,1.406401,0.625792,"1.07 [0.81, 1.41]"
2,Merge Hazard,Verification (V),0.840080,0.645681,1.093009,0.194389,"0.84 [0.65, 1.09]"
3,Merge Hazard,Log(PR Size),0.932051,0.872924,0.995183,0.035342,"0.93* [0.87, 1.00]"
4,Merge Hazard,Observations,261.000000,NaN,NaN,NaN,261
5,Close Hazard,Context (C),1.723025,0.947723,3.132576,0.074432,"1.72 [0.95, 3.13]"
6,Close Hazard,Specificity (S),0.631869,0.344058,1.160437,0.138813,"0.63 [0.34, 1.16]"
7,Close Hazard,Verification (V),1.767973,1.025761,3.047227,0.040211,"1.77* [1.03, 3.05]"
8,Close Hazard,Log(PR Size),0.828583,0.719614,0.954053,0.008953,"0.83** [0.72, 0.95]"
9,Close Hazard,Observations,261.000000,NaN,NaN,NaN,261
